# 03 — Handling class imbalance: weighting vs SMOTE vs undersampling

**Question:** with ~0.17 % fraud, which imbalance strategy gives the best precision/recall trade-off?

**Protocol (same as `python -m src.train`)**
- One stratified 70/15/15 split (`src.preprocess.split_data`, seed 42).
- Every strategy is fitted on the **training** split only. Scaling for SMOTE's neighbour search and all
  resampling are fitted on training rows, never on validation/test rows.
- Each model's decision threshold is chosen on the **validation** split (highest recall with precision ≥ 0.80).
- The **test** split is evaluated once, after those choices.

> This notebook was rebuilt: the previous file was corrupted (an empty notebook JSON followed by
> a fragment of the original). Its code was recovered from the fragment and moved onto the shared
> protocol, replacing the earlier 80/20 split that had no validation set. Run it top to bottom to
> produce results; none are stored in the file.

In [ ]:
import os
import sys
from pathlib import Path

# Make the repository importable whether Jupyter starts in notebooks/ or the repo root.
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config, evaluate, models, preprocess

# Set BEYONDACCURACY_QUICK=1 for a fast smoke run (fewer trees); leave unset for the real run.
XGB_PARAMS = {"n_estimators": 25} if os.environ.get("BEYONDACCURACY_QUICK") else None
print("Dataset path:", config.DATA_PATH)

In [ ]:
df = preprocess.load_dataset()          # raises a helpful error if data/creditcard.csv is missing
X_train, X_val, X_test, y_train, y_val, y_test = preprocess.split_data(df)

pd.DataFrame(
    {"rows": [len(X_train), len(X_val), len(X_test)],
     "fraud": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())]},
    index=["train (fit models)", "validation (choose threshold & model)", "test (final report only)"],
).assign(fraud_rate=lambda t: t["fraud"] / t["rows"])

## The accuracy trap
A model that never flags anything is already extremely "accurate" on this data.

In [ ]:
majority_accuracy = 1 - y_test.mean()
print(f"Accuracy of 'always predict legitimate' on the test split: {majority_accuracy:.4%}")
print(f"Fraud cases it catches: 0 of {int(y_test.sum())}")

## Train every strategy and evaluate with a validation-chosen threshold

In [ ]:
rows = {}
fitted = {}
for name in models.EXPERIMENTS:
    model = models.fit_experiment(name, X_train, y_train, params=XGB_PARAMS)
    p_val = model.predict_proba(X_val)[:, 1]
    p_test = model.predict_proba(X_test)[:, 1]
    thr = evaluate.select_threshold(y_val, p_val)["threshold"]          # validation only
    tuned = evaluate.compute_metrics(y_test, p_test, thr)               # test, final report
    default = evaluate.compute_metrics(y_test, p_test, 0.5)             # for contrast
    cm = tuned["confusion_matrix"]
    rows[name] = {
        "threshold": thr,
        "accuracy": tuned["accuracy"],
        "precision": tuned["precision"],
        "recall": tuned["recall"],
        "f1": tuned["f1"],
        "avg_precision": tuned["average_precision"],
        "roc_auc": tuned["roc_auc"],
        "false_alarms": cm["fp"],
        "missed_fraud": cm["fn"],
        "recall_at_0.5": default["recall"],
        "precision_at_0.5": default["precision"],
    }
    fitted[name] = model

results_df = pd.DataFrame(rows).T
results_df

## Visual comparison
Bars show test-set precision and recall at each model's validation-chosen threshold.

In [ ]:
ax = results_df[["recall", "precision"]].plot(kind="bar", figsize=(10, 5))
ax.set_title("Recall vs precision by imbalance-handling strategy (test set, validation-tuned thresholds)")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=15)
plt.tight_layout()
config.ASSETS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.ASSETS_DIR / "imbalance_comparison.png", dpi=130)
plt.show()

## How to read the table
- Compare **average precision**, **precision** and **recall** together; accuracy barely moves between strategies.
- Resampled models (SMOTE, undersampling) produce distorted probabilities. That is why the threshold is
  re-selected on validation data for every model, and why `recall_at_0.5` / `precision_at_0.5` can look very
  different from the tuned columns.
- Aggressive undersampling usually raises recall at a large precision cost (many false alarms).
- Differences between strategies are measured on a small number of test-set fraud cases
  (see the table at the top), so small gaps are within noise. Do not over-interpret them.